# Aid Activity

**Project:** Pali Watch

**Notebook:** `aid_activity.ipynb`

**Author:** Yuseof J

**Created:** 11/11/2026 | **Last updated:** 14/11/2026

## Purpose
This file maps the more granular components of aid activity and coverage in Palestine. It dives into actual timeframes of programs as well as specific activity of aid programs.

## Inputs

- `data/raw/boundaries/`: HDX admin boundaries (COD-AB)

## Outputs
- `data/processed/[filename].csv`: what it contains

## Notes and caveats
- Data limitations: iati data lacks location for 30-60% of activities. Additionally, it only represents aid activity that is reported, it is not an exhaustive list of aid operations
- as of 11/14/2026, setting this component of the project aside to progress with stablishing the rest of the project pipeline

In [ ]:
import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

CRS = 'EPSG:4326'

### Data imports / exports

In [ ]:
# import paths
PATH_TO_ADMIN_BOUNDARIES = f'{RAW}/pse_admin_boundaries.shp/pse_admin2.shp'
PATH_TO_AID_ACTIVITY_GEO = f'{RAW}/iati-activity-locations-in-state-of-palestine.csv'
PATH_TO_AID_ACTIVITY_DETAILS = f'{RAW}/iati-activities-in-state-of-palestine-no-location-information.csv'

# export paths
PATH_TO_GEOLOCATED_EVENTS_GPKG = f'{PROCESSED}/acled/conflict_events_geolocated.gpkg'
PATH_TO_GEOLOCATED_EVENTS_DF = f'{PROCESSED}/acled/conflict_events_geolocated.csv'

# import data
df_aid_activity_loc = pd.read_csv(PATH_TO_AID_ACTIVITY_GEO)
df_aid_activity_details = pd.read_csv(PATH_TO_AID_ACTIVITY_DETAILS)
gdf_boundaries = gpd.read_file(PATH_TO_ADMIN_BOUNDARIES)

In [ ]:
df_aid_activity_details.head()

## EDA - Sector and Aid Type

What columns can be used to determine the secotr / type of aid (e.g. food, cash, development, healthcare, etc)?

'reporting' column gives us the organization name 

In [ ]:
#df_aid_activity_loc.reporting.unique()

'title' column gives us program names / short descriptions, but cannot be used as a standardized categorization of aid type

In [ ]:
# pd.DataFrame(df_aid_activity_loc.title.unique()).to_string()

In [ ]:
df_aid_activity_details.sector_group.unique()

In [ ]:
violence_related_sector_groups = ['VIII.2. Reconstruction and Rehabilitation', 
                                    'VIII.1. Emergency Response', 
                                    'I.5.b. Conflict, peace and security', 
                                    'VIII.3. Disaster Preparedness',
                                    '920'
                                 ]

In [ ]:
mask_violence_related_aid = df_aid_activity_details.sector_group.isin(violence_related_sector_groups)
df_aid_activity_details[mask_violence_related_aid].groupby('sector_group').sector_code.unique()

## Preprocessing

Isolate location and primary keys from location coverage

In [ ]:
df_aid_activity_loc_ids = df_aid_activity_loc[['reporting', 'reporting_ref', 'slug', 'day_start', 'day_end', 'location_longitude', 'location_latitude']].drop_duplicates()
df_aid_activity_loc_ids.head()

Isolate coverage details from non-location version of dataset

In [ ]:
df_aid_activity_details_ids = df_aid_activity_details[['reporting', 'reporting_ref', 'slug', 'day_start', 'day_end', 'sector_group', 'sector_code', 'sector_percent']].drop_duplicates()
df_aid_activity_details_ids.head()

Join aid details to aid locations

In [ ]:
df_aid_activity_full = df_aid_activity_details_ids.merge(df_aid_activity_loc_ids,
                                      how='left',
                                      on=['reporting', 'reporting_ref', 'slug', 'day_start', 'day_end']
                                     )

In [ ]:
df_aid_activity_full.head(20)

match rate

In [ ]:
df_aid_activity_full.location_latitude.isna().value_counts(normalize=True)

In [ ]:
df_aid_activity_full[df_aid_activity_full.year_end>2025].location_latitude.isna().value_counts(normalize=True)

QA checks

In [ ]:
# does the same aid (start and end date) map to two different locations? If so, I can't tell which location received which aid...

In [ ]:
mask_geo_ambiguous = df_aid_activity_full.
df_aid_activity_full.groupby('reporting', 'reporting_ref', 'slug', 'day_start', 'day_end')

In [ ]:
# format date columns
df_aid_activity_full['day_start'] = pd.to_datetime(df_aid_activity_full['day_start'], errors='coerce')
df_aid_activity_full['day_end'] = pd.to_datetime(df_aid_activity_full['day_end'], errors='coerce')

# create date grouping columns
df_aid_activity_full['year_start'] = df_aid_activity_full['day_start'].dt.year
df_aid_activity_full['yearmo_start'] = df_aid_activity_full['day_start'].dt.to_period('M').astype(str)

df_aid_activity_full['year_end'] = df_aid_activity_full['day_end'].dt.year
df_aid_activity_full['yearmo_end'] = df_aid_activity_full['day_end'].dt.to_period('M').astype(str)

## Geoprocessing

In [ ]:
# convert event coordinates into spatial points
gdf_aid_activity = gpd.GeoDataFrame(
    df_aid_activity,
    geometry=gpd.points_from_xy(
        df_aid_activity['location_longitude'],
        df_aid_activity['location_latitude']
    ),
    crs=CRS
)

In [ ]:
# check and align CRSs
print("Events CRS:", gdf_aid_activity.crs)
print("Admin CRS:", gdf_boundaries.crs)
gdf_aid_activity = gdf_aid_activity.to_crs(gdf_boundaries.crs) # failsafe -- convert events CRS to admin boundaries CRS if not already aligned
assert(gdf_aid_activity.crs == gdf_boundaries.crs)
print('Confirmed CRSs are aligned')

In [ ]:
# perform spatial join (assign event coordinates to an administrative boundary)
gdf_aid_activity_joined = gpd.sjoin(
    gdf_aid_activity,
    gdf_boundaries[['adm1_name', 'adm1_pcode', 'adm2_name', 'adm2_pcode', 'geometry']],
    how='left',
    predicate='within'
)

In [ ]:
print("Total events:", len(gdf_aid_activity_joined))
print("Matched events:", gdf_aid_activity_joined['adm1_name'].notna().sum())
print("Unmatched events:", gdf_aid_activity_joined['adm1_name'].isna().sum())

In [ ]:
unmatched = gdf_aid_activity_joined[gdf_aid_activity_joined['adm1_name'].isna()]
if len(unmatched) > 0:
    display(unmatched.head())
else:
    print('All events matched successfully!')

In [ ]:
gdf_aid_activity_joined.head()

## Governorate Aggregates

### Active Org Count

In [ ]:
Keep in mind, this filtering step would theoretically be done in the applet UI (i.e. user filters for date range, and this code is run to produce the number of active orgs)
Additionally, I can run this code to create static reports / analyses

Conflict events are different than other risks. They are "sudden" (which is to say, they happen in a moment and then they end. This is slightly different than something like food insecurity. Aspects of violent conflict are similar -- there is a buildup, signs that things are getting worse,
the moment things escalate (i.e. violence happens), and then a de-escalation. But it happens more suddenly than the need for food, for money, for healthcare. I need to see this in parallel. Compare aspects of risk_conflict and risk_food_insecurity. Is risk_conflict that much different in 
terms of predictability, buildup, prevention, resolution?

Precursor to this: do orgs currently respond to violent conflicts? is there an aid type that addresses this such that I can even identify gaps?

Sprint 1:
    
    Questions we want to answer:
        - before the event was aid in place?
        - during the event, was there aid in place?
        - after the event, was there aid in place?

Beyond:
    Can we predict where/when future events will be? Can we mobilize aid accordingly?
        - for future event, will the aid be there before/during/after? Can we ensure it is ready before?


yearmo = '10/2026'

for each yearmo
    
    # programs active per yearmo
    active_programs = (title yearmo_start <= yearmo) AND (title yearmo_end >= yearmo) 
    
    # unique orgs among active programs
    active_orgs = active_programs.reporting.unique()

    # sector + population coverage (maybe breack pop cov into its own section if below gets too complicated)
    active_programs.groupby(sector).agg(
        size() # number of programs per sector
        reporting.nunique() # number of unique orgs per sector
        lamda x: sum(population covered by each program) # total population coverage
        lambda y: above lambda / total pop for that area = % pop served (caveat, assumes everyone in that area is in need instead of denominator being subset of people in population in need: Can I figure this out somehow? i.e. isolate people with given needs? eg are people with low income the ones receiving the program aid?)
        
        

### Sector Coverage

### Population Coverage

Plan:

- figure out how to determine the type of aid (i.e. is this relevant to settler violence? to healthcare? food insecurity?)
- do I need to bring in the non-location dataset to do this?
- IMPORTANT: IS THERE ANY AID DESIGNATED TO CONFLICT-RELATED PREVENTION/RESOLUTION (ESP SETTLER VIOLENCE), IF NOT, GAP ANALYSIS CAN'T REALLY BE PERFORMED, BUT! CAN STILL SHOW CONFLICT AND SUGGEST AREAS FOR AID AND TYPES OF AID 
- aggregate per governorate (active org count, distinct sectors covered, % of population served or per capita coverage)
- build coverage score (org count, sector breadth, per capita coverage, etc) and normalize this on a 0-1 scale
- merge this into a master gdf dataset